# IOAI — 2025 Contest Underfitting Cv (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.system('pip -q install timm')
if not os.path.exists('data/train.csv'):
    os.makedirs('data', exist_ok=True)
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-underfitting-cv', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Underfitting CV · 모범답안

**통찰: "underfitting" = 제공 모델이 덜 학습됨.** 밑바닥 학습은 230장으론 불가능하지만, 제공된 `model.pt`는
이미 102클래스 head 를 갖고 있으니 **그 위에서 계속 학습(warm-start 미세조정)** 하면 underfitting 이 완화됩니다.
230장이 적으므로 **데이터 증강(RandAugment)** + **낮은 학습률**(망각 방지) + **라벨 스무딩**으로 과적합을 누릅니다.
102클래스 head 를 *유지*(reset 금지 — 학습에 없는 ~79개 클래스도 예측해야 함). 전처리는 베이스라인과 동일(정규화 없음).
실제 캐글 정확도: 베이스라인 ≈ 0.33 → **모범답안 ≈ 0.67**(리더보드 1위 ≈ 0.788).

In [ ]:
import torch, timm, pandas as pd, numpy as np, os, time
from PIL import Image
from torch import nn
import torchvision.transforms as T
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
tr = pd.read_csv('data/train.csv')   # path, class (원래 클래스 id 0..101 중 23개)

def load_model():
    m = timm.create_model('tiny_vit_5m_224', num_classes=102)
    sd = torch.load('data/model.pt', map_location='cpu', weights_only=False)
    m.load_state_dict({k[len('model.'):]: v for k, v in sd.items()})
    return m.to(dev)

tf_eval = T.Compose([T.Resize((224, 224)), T.ToTensor()])
tf_aug  = T.Compose([T.Resize((224, 224)), T.RandomHorizontalFlip(),
                     T.RandAugment(num_ops=2, magnitude=7), T.ToTensor()])


In [ ]:
# warm-start 미세조정 (102 head 유지, 낮은 LR + 증강 + 라벨 스무딩)
m = load_model(); y = torch.tensor(tr['class'].values); paths = tr['path'].values
opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
t = time.time()
for ep in range(40):
    m.train(); perm = np.random.permutation(len(tr))
    for i in range(0, len(perm), 32):
        idx = perm[i:i+32]
        xb = torch.stack([tf_aug(Image.open('data/train_images/'+paths[j]).convert('RGB')) for j in idx]).to(dev)
        opt.zero_grad(); lossf(m(xb), y[idx].to(dev)).backward(); opt.step()
print('fine-tune 40ep done', round(time.time()-t, 1), 's')


In [ ]:
# 전체 test 예측 → submission.csv
test_files = sorted(os.listdir('data/test_images'))
m.eval(); out = []
with torch.no_grad():
    for i in range(0, len(test_files), 128):
        xb = torch.stack([tf_eval(Image.open('data/test_images/'+f).convert('RGB')) for f in test_files[i:i+128]]).to(dev)
        out.append(m(xb).argmax(1).cpu().numpy())
pred = np.concatenate(out)
pd.DataFrame({'id': test_files, 'class': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(test_files), '| 예측 클래스 수', len(set(pred)),
      '\n→ Submissions 탭에서 캐글 채점 (모범답안 ≈ 0.67)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)